In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf 
from statsmodels.iolib.summary2 import summary_col

### First Task: Relationship between religiosity and economic development
I estimate the  relationship between intensity of religiosity at the local level and economic development during the Second Industrial Revolution, where the religiosity is measured by the share of refractory clergy.

The dataset used contains information contains department-level information before and after the Second Industrial Revolution.

In [ ]:
def var_labels(data_path):
    stata_reader = pd.read_stata(data_path, iterator=True)
    labels_dict = stata_reader.variable_labels()
    data_dictionary = pd.DataFrame(columns=["Variable", "Label"], 
                                   data=list(labels_dict.items()))
    return data_dictionary

In [ ]:
dept_data  = pd.read_stata("Data/1.dept_dataset.dta")
dept_labels = var_labels("Data/1.dept_dataset.dta")

We can then create a DataFrame from the labels dictionary. Through Data Wrangles plugin in VScode we can replicate a Stata-like experience when it comes to exploring variables and their labels. Just go in the Jupyter tab, and click on the Dataframe created below.

I first replicate Table 2 from the paper, which shows the association between the intensity of religiosity on different variable before and after the second industrial revolution.

In [ ]:
model1 = smf.ols(formula="share_ind_work1901 ~ share_refractory", data=dept_data).fit(cov_type="HC1")
model2 = smf.ols(formula="machines_pc ~ share_refractory", data=dept_data).fit(cov_type="HC1")
model3 = smf.ols(formula="share_ind_work1866 ~ share_refractory", data=dept_data).fit(cov_type="HC1")
model4 = smf.ols(formula="steam_engine_pc ~ share_refractory", data=dept_data).fit(cov_type="HC1")

table2 = summary_col([model1, model2, model3, model4], 
            stars=True, 
            float_format="%.3f",
            info_dict={ "Obs" : lambda x : f"{x.nobs:.0f}"},
            model_names=["Share ind. workers 1901", "Machines pc, 1891", "Share ind. workers 1866", "Steam eng. pc, 1840"],
            regressor_order=["share_refractory"],
            drop_omitted=True,
            )
table2

Table 3 includes sevaral controls

In [ ]:
regressors1 = ["share_refractory","lnpop1901", "mean_lntemp", "mean_lnprec", "wheat_suit","lnproto_ind"]
regressors2 = ["share_refractory","lnpop1891", "mean_lntemp", "mean_lnprec", "wheat_suit","lnproto_ind"]
regressors3 = regressors1 + ["lndist_Paris", "pays_elections"]
regressors4 = regressors2 + ["lndist_Paris", "pays_elections"]
regressors5 = regressors3 + ["lnsub_dept", "enrol_rate1891"]
regressors6 = regressors4 + ["lnsub_dept", "enrol_rate1881"]

In [ ]:
model1 = smf.ols(formula=f"share_ind_work1901 ~ {"+".join(regressors1)}", data=dept_data).fit(cov_type="HC1")
model2 = smf.ols(formula=f"machines_pc ~ {"+".join(regressors2)}", data=dept_data).fit(cov_type="HC1")
model3 = smf.ols(formula=f"share_ind_work1901 ~ {"+".join(regressors3)}", data=dept_data).fit(cov_type="HC1")
model4 = smf.ols(formula=f"machines_pc ~ {"+".join(regressors4)}", data=dept_data).fit(cov_type="HC1")
model5 = smf.ols(formula=f"share_ind_work1901 ~ {"+".join(regressors5)}", data=dept_data).fit(cov_type="HC1")
model6 = smf.ols(formula=f"machines_pc ~ {"+".join(regressors6)}", data=dept_data).fit(cov_type="HC1")

reg_order = ["share_refractory","lnpop1901","lnpop1891", "mean_lntemp", "mean_lnprec", "wheat_suit","lnproto_ind","lndist_Paris", "pays_elections", "lnsub_dept", "enrol_rate1881"]

In [ ]:
summary_col([model1, model2, model3, model4, model5, model6], 
            float_format="%.3f", 
            stars=True, 
            regressor_order=reg_order,
            drop_omitted=True,
            info_dict= {"Obs" : lambda x : f"{x.nobs:.0f}"},
            model_names=["Share ind.workers, 1901", "Machinespc, 1891"] * 3)

### Second Task: Driving mechanism behind the relationship
I reproduce Table 7 columns 4 and 7 and Figure 2 from the paper. The table indicates that cantons with higher levels of religiosity have experienced a greater increase in the proportion of Catholic schools. Figure 2 illustrates the evolution of the share of Catholic and secular schools in both the private and public sectors.

In [ ]:
cantons_data = pd.read_stata("Data/2b.canton_dataset_schooling.dta")
cantons_labels = var_labels("Data/2b.canton_dataset_schooling.dta")

schools_data = pd.read_stata("Data/cath_sec_schools.dta")
schools_labels = var_labels("Data/cath_sec_schools.dta")

The dependent variables are the share of catholic schools and the growth share of catholic schools, and standard errors are clustered at the district level.

In [ ]:
regressors1 = ["share_refractory", "lnpopulation","lnstud_ps", "lnschools_total", "department_id"] 
regressors2 = ["share_refractory", "gr_population", "share_cath_schools_l", "lnstud_ps_l", "lnschools_total_l", "wheat_suit", "lndist_Paris", "lnsubs", "department_id"]

When I initially run the regression, statsmodels raised a value error due to a mismatch between the number of rows used in the estimation and the length of the clustering column (district_id) specified in cov_kdws. This happens because statsmodels automatically drops any rows that presents missing values for the variables included in the equation, but it doesn't apply the same filter to the clustering variables.

To solve this, I retrived the indices of the rows retained by the model, and used them to filter the district_id column before fitting.

In [ ]:
model1 = smf.ols(formula=f"share_cath_schools ~ + {"+".join(regressors1[:-1])} + C(department_id)", data=cantons_data.query("year == 1894"))
clusters = cantons_data.query("year == 1894").loc[model1.data.row_labels, "district_id"]
col4 = model1.fit(cov_type="cluster", cov_kwds={"groups" : clusters})


model2= smf.ols(formula=f"gr_share_cath_schools ~ + {"+".join(regressors2[:-1])} + C(department_id)", data=cantons_data.query("year==1894")) 
clusters = cantons_data.query("year == 1894").loc[model2.data.row_labels, "district_id"]
col7 = model2.fit(cov_type="cluster", cov_kwds={"groups" : clusters})

In [ ]:
summary_col([col4, col7],
            regressor_order=["share_refractory"], 
            drop_omitted=True,
            stars=True,
            float_format="%.3f",
            info_dict={"N" : lambda x : f"{x.nobs:.0f}"},
            model_names=["Share Catholic Schools 1894", "Growth Share Catholic Schools 1873-1894"])

In [ ]:
schools_data = schools_data.assign(
    tot_cath_schools = schools_data["schools_priv_cath"] + schools_data["schools_pub_cath"],
    tot_lay_schools =
schools_data["schools_priv_lay"] + schools_data["schools_pub_lay"]
)

schools_data_years = schools_data.groupby("year").sum().reset_index()
schools_data_years.columns = ["year", "department_id", "Public Secular Schools", "Public Catholic Schools",
                                          "Private Secular Schools", "Private Catholic Schools",
                                          "Total Catholic Schools", "Total Secular Schools"]
schools_data_years

In [ ]:
fig, (ax1,ax2) = plt.subplots(1,2, figsize=(15,5), dpi=300)
plt.style.use("seaborn-v0_8-deep")
line_styles = ["dotted", "dashed", "solid"]


for key,val in dict(zip(schools_data_years.iloc[:, np.r_[3,5,6]].columns, line_styles)).items(): #this selects only catholic schools columns
    ax1.plot(schools_data_years["year"], 
             schools_data_years[key],
             label=key,
             ls = val)

for key,val in dict(zip(schools_data_years.iloc[:, np.r_[2,4,7]].columns, line_styles)).items(): #this selects only catholic schools columns
    ax2.plot(schools_data_years["year"], 
                schools_data_years[key],
                label=key,
                ls = val)

ax1.set_yticks(np.arange(0,25001, 5000))
ax2.set_yticks(np.arange(0,80001, 20000))
ax1.legend()
ax2.legend()
ax1.set_title("Number of catholic schools")
ax2.set_title("Number of secualr schools")
plt.suptitle("Catholic and Secular Schools in the Public and Private Sector, 1866-1901",
             fontsize=16, fontfamily="Times")
ax1.yaxis.set_major_formatter("{x:,.0f}")
ax2.yaxis.set_major_formatter("{x:,.0f}")


plt.show()

### Task 3: Channels

In [ ]:
dept_data
cohort_df = pd.read_stata("Data/3b.cohort_dataset.dta")
cohort_labels = var_labels("Data/3b.cohort_dataset.dta")

In [ ]:
controls = ["enrol_rate", "lnstud_ps", "lnschools_total"]
order = ["share_cath_schools", "share_cath_stud"]
# share_cohort_mod_workers

reg1 = (smf.ols(formula=f"share_cohort_mod_workers ~ share_cath_schools +  + lnpop + C(department_id) + C(year)",
               data=cohort_df)
               .fit(cov_type="cluster",
                    cov_kwds={"groups" : cohort_df["department_id"]}))
               
reg2 = (smf.ols(formula=f"share_cohort_mod_workers ~ share_cath_schools + lnpop + {"+".join(controls)} + C(department_id) + C(year)",
               data=cohort_df)
               .fit(cov_type="cluster",
                    cov_kwds={"groups" : cohort_df["department_id"]}))
               
reg3 = (smf.ols(formula=f"share_cohort_mod_workers ~ share_cath_stud + lnpop +  C(department_id) + C(year)",
               data=cohort_df)
               .fit(cov_type="cluster",
                    cov_kwds={"groups" : cohort_df["department_id"]}))

reg4 = (smf.ols(formula=f"share_cohort_mod_workers ~ share_cath_stud + lnpop + {"+".join(controls)} + C(department_id) + C(year)",
               data=cohort_df)
               .fit(cov_type="cluster",
                    cov_kwds={"groups" : cohort_df["department_id"]}))

In [21]:
info = {"   Schooling controls" : lambda x: "✓" if "lnstud_ps" in x.model.exog_names else "",
        "  Department FE" : lambda x : "✓" if any("C(department_id)" in d for d in x.model.exog_names) else "",
        " Cohort FE" : lambda x  : "✓" if any("C(year)" in y for y in  x.model.exog_names) else "",
        "R2" : lambda x : f"{x.rsquared:.2f}",
        "Obs" : lambda x : f"{x.nobs:.0f}"}

table12 = summary_col([reg1, reg2, reg3, reg4],
            regressor_order=order,
            drop_omitted=True,
            float_format="%.3f",
            stars=True,
            info_dict=info,
            model_names=["(1)", "(2)", "(3)", "(4)"],
            include_r2=False)
table12.add_title("Religiously educated cohorts less likely to be employed in innovative sectors")
table12.add_text("Dependent variable: share of workers in innovative sectors")
table12

,(1),(2),(3),(4)
share_cath_schools,-0.384**,-0.381**,,
,(0.150),(0.154),,
share_cath_stud,,,-0.206*,-0.239**
,,,(0.118),(0.108)
Schooling controls,,✓,,✓
Department FE,✓,✓,✓,✓
Cohort FE,✓,✓,✓,✓
Obs,249,249,249,249
R2,0.97,0.97,0.97,0.97


In [23]:
print(table12.as_latex())

\begin{table}
\caption{Religiously educated cohorts less likely to be employed in innovative sectors}
\label{}
\begin{center}
\begin{tabular}{lllll}
\hline
                      & (1)      & (2)      & (3)     & (4)       \\
\hline
share\_cath\_schools  & -0.384** & -0.381** &         &           \\
                      & (0.150)  & (0.154)  &         &           \\
share\_cath\_stud     &          &          & -0.206* & -0.239**  \\
                      &          &          & (0.118) & (0.108)   \\
   Schooling controls &          & ✓        &         & ✓         \\
  Department FE       & ✓        & ✓        & ✓       & ✓         \\
 Cohort FE            & ✓        & ✓        & ✓       & ✓         \\
Obs                   & 249      & 249      & 249     & 249       \\
R2                    & 0.97     & 0.97     & 0.97    & 0.97      \\
\hline
\end{tabular}
\end{center}
\end{table}
\bigskip
Standard errors in parentheses. \newline 
* p<.1, ** p<.05, ***p<.01 \newline 
Dependent var